# 06 Backward and Gradients

This notebook audits backward capture: `backward_ready`, gradient saving, `GradFn` and `GradFnCall` records, and backward validation. A `GradFn` is TorchLens metadata for one PyTorch autograd node; a `GradFnCall` is one recorded backward hook/call event.

The model returns a scalar loss so a real `.backward()` call is simple and deterministic.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "audit":
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo root: {REPO_ROOT}")

In [ ]:
import inspect

import torch
from torch import nn
import torchlens as tl
from torchlens.validation import validate_backward_pass

torch.manual_seed(29)


class BackwardNet(nn.Module):
    """Small scalar-output model for backward capture."""

    def __init__(self) -> None:
        """Initialize layers."""

        super().__init__()
        self.stem = nn.Linear(3, 4)
        self.head = nn.Linear(4, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the model and return a scalar loss.

        Parameters
        ----------
        x:
            Input tensor.

        Returns
        -------
        torch.Tensor
            Scalar loss-like output.
        """

        hidden = torch.relu(self.stem(x))
        return self.head(hidden).pow(2).mean()


model = BackwardNet().eval()
x = torch.randn(5, 3, requires_grad=True)
print(f"tl.trace has backward_ready: {'backward_ready' in inspect.signature(tl.trace).parameters}")
print(f"Trace.log_backward signature: {inspect.signature(tl.Trace.log_backward)}")

`backward_ready=True` keeps the graph connected for a later backward pass. `gradients_to_save="all"` asks TorchLens to retain gradient payloads where it can.

In [ ]:
trace = tl.trace(
    model,
    x,
    backward_ready=True,
    gradients_to_save="all",
)
loss_layer = trace[trace.output_layers[0]]
print(f"loss layer: {loss_layer.layer_label}")
print(f"loss value: {loss_layer.out.item():.6f}")
print(f"loss requires_grad: {loss_layer.out.requires_grad}")
print(f"saved grad ops before backward: {len(trace.saved_grad_ops)}")

Calling `Trace.log_backward(loss)` runs PyTorch backward and records TorchLens backward metadata.

In [ ]:
trace.log_backward(loss_layer.out)
print(f"input .grad shape: {tuple(x.grad.shape)}")
print(f"input .grad norm: {x.grad.norm().item():.6f}")
print(f"grad fns: {len(trace.grad_fns)}")
print(f"grad fn calls: {len(trace.grad_fn_calls)}")
print(f"saved grad ops: {len(trace.saved_grad_ops)}")

Forward `Op` records can now expose saved gradients. This cell prints the first few gradient-bearing operations.

In [ ]:
for label in list(trace.saved_grad_ops.keys())[:5]:
    op = trace[label]
    grad = op.grad
    grad_shape = tuple(grad.shape) if isinstance(grad, torch.Tensor) else None
    print(f"{label:16s} func={op.func_name:8s} grad_shape={grad_shape} saved={grad is not None}")

`GradFn` records describe autograd nodes. `GradFnCall` records describe individual backward call events, including whether a gradient payload was saved.

In [ ]:
for grad_fn in list(trace.grad_fns)[:4]:
    print(
        f"GradFn {grad_fn.label:18s} class={grad_fn.class_name:18s} "
        f"op_label={grad_fn.op_label} calls={list(grad_fn.calls)}"
    )

print("--- calls ---")
for grad_call in list(trace.grad_fn_calls)[:4]:
    print(
        f"GradFnCall {grad_call.call_label:18s} "
        f"parent={grad_call.label:18s} saved={grad_call.is_saved}"
    )

Forward/backward linkage summary:

| Object | What it is | Where to inspect it |
|---|---|---|
| Forward `Op.grad_fn_handle` | The live PyTorch autograd handle captured during forward. | `op.grad_fn_handle` / `op.grad_fn_class_name` |
| TorchLens `GradFn` | TorchLens metadata for an autograd node. | `trace.grad_fns[...]` |
| Saved forward gradient | Tensor gradient retained on a forward op after backward. | `op.grad` |

In [ ]:
linked_grad_fn = next(grad_fn for grad_fn in trace.grad_fns if grad_fn.op_label is not None)
linked_op = trace[linked_grad_fn.op_label]
print(f"forward op: {linked_op.layer_label} func={linked_op.func_name}")
print(f"op.grad_fn_class_name: {linked_op.grad_fn_class_name}")
print(f"TorchLens GradFn: {linked_grad_fn.label} class={linked_grad_fn.class_name}")
print(f"saved op.grad present: {isinstance(linked_op.grad, torch.Tensor)}")

`validate_backward_pass` compares TorchLens backward capture with ordinary PyTorch autograd. We import it from `torchlens.validation` to avoid the deprecated top-level shim.

In [ ]:
validation_model = BackwardNet().eval()
validation_model.load_state_dict(model.state_dict())
validation_x = x.detach().clone().requires_grad_(True)
valid = validate_backward_pass(
    validation_model,
    validation_x,
    loss_fn=lambda output: output,
    random_seed=29,
    validate_metadata=False,
)
print(f"validate_backward_pass: {valid}")

> NOTE: The glossary names `GradFn.grad_fn_class_name` and `Op.has_saved_gradient`, but this build exposes the public class name on `GradFn.class_name` and uses `op.grad is not None` as the executable saved-gradient check. Forward `Op` records still expose `grad_fn_class_name`.

## Backward Selectors, Helpers, and Observers
Backward hooks are installed at trace time through `hooks=(selector, helper)`. This build exports `tl.grad_fn`, `tl.bwd_hook`, `tl.grad_clip`, `tl.grad_noise`, and `tl.grad_clamp`; `tl.grad_fn_label` is not exported. Tensor-gradient helpers `tl.grad_zero` and `tl.grad_scale` are present, but mounting them through the public trace hook path raises in this build, so this cell records that gap instead of faking a result.

In [ ]:
def run_backward_helper(name: str, selector: object, helper: object) -> None:
    """Trace with one backward hook and print the resulting input gradient."""
    hooked_x = torch.randn(3, 3, requires_grad=True)
    hooked_log = tl.trace(
        model,
        hooked_x,
        backward_ready=True,
        gradients_to_save="all",
        intervention_ready=True,
        hooks=(selector, helper),
    )
    hooked_log.log_backward(hooked_log[hooked_log.output_layers[0]].out)
    print(f"{name}: input_grad_norm={hooked_x.grad.norm().item():.6f}")
    for label in list(hooked_log.saved_grad_ops.keys())[:3]:
        grad = hooked_log.saved_grad_ops[label].grad
        print(f"  {label}: grad_sum={grad.sum().item():.6f}")


run_backward_helper(
    "tl.grad_clamp on tl.grad_fn(type='mean')", tl.grad_fn(type="mean"), tl.grad_clamp(max=0.05)
)
run_backward_helper(
    "tl.grad_noise on tl.grad_fn(type='mean')",
    tl.grad_fn(type="mean"),
    tl.grad_noise(0.01, seed=11),
)
run_backward_helper(
    "tl.grad_clip on tl.grad_fn(type='mean')", tl.grad_fn(type="mean"), tl.grad_clip(0.1)
)


def quarter_backward(
    grad_input: tuple[torch.Tensor | None, ...], *, hook: object
) -> tuple[torch.Tensor | None, ...]:
    """Scale tuple-shaped backward input gradients."""
    del hook
    return tuple(None if grad is None else grad * 0.25 for grad in grad_input)


try:
    run_backward_helper(
        "tl.bwd_hook custom scale", tl.grad_fn(type="mean"), tl.bwd_hook(quarter_backward)
    )
except Exception as exc:
    print(f"> NOTE: tl.bwd_hook public mount gap: {type(exc).__name__}: {exc}")
print(f"tl.grad_fn_label exported: {hasattr(tl, 'grad_fn_label')}")

for helper in [tl.grad_zero(), tl.grad_scale(0.5)]:
    try:
        probe_x = torch.randn(2, 3, requires_grad=True)
        probe = tl.trace(
            model,
            probe_x,
            backward_ready=True,
            gradients_to_save="all",
            intervention_ready=True,
            hooks=(tl.func("linear"), helper),
        )
        probe.log_backward(probe[probe.output_layers[0]].out)
        print(f"{helper.helper_name}: input_grad_norm={probe_x.grad.norm().item():.6f}")
    except Exception as exc:
        print(f"> NOTE: {helper.helper_name} public mount gap: {type(exc).__name__}: {exc}")

In [ ]:
tap = tl.tap(tl.contains("linear"), direction="both")
with tl.record_span("linear_probe", direction="both"):
    observed = tl.trace(model, torch.randn(2, 3), intervention_ready=True, hooks=tap)
    observed.log_backward(observed[observed.output_layers[0]].out)
print(f"tap records: {len(tap.records)}")
for record in tap.records:
    value = record.value
    shape = tuple(value.shape) if isinstance(value, torch.Tensor) else None
    print(
        f"{record.direction}: site={record.site_label} grad_kind={record.grad_kind} "
        f"shape={shape} spans={record.span_names}"
    )
print(f"trace observer spans: {[span['name'] for span in observed.observer_spans]}")

## 🔧 Sandbox

Mini-experiment: change `sandbox_batch`, `sandbox_loss_kind`, and `sandbox_save_grads`. Expected observation: gradient norm and saved-gradient counts move with batch/loss/save settings.

In [ ]:
sandbox_batch = 3
sandbox_loss_kind = "sum"
sandbox_save_grads = "all"
sandbox_x = torch.randn(sandbox_batch, 3, requires_grad=True)
sandbox_trace = tl.trace(
    model,
    sandbox_x,
    backward_ready=True,
    gradients_to_save=sandbox_save_grads,
)
sandbox_output = sandbox_trace[sandbox_trace.output_layers[0]].out
sandbox_loss = sandbox_output if sandbox_loss_kind == "sum" else sandbox_output.abs()
sandbox_trace.log_backward(sandbox_loss)
print(f"batch: {x.shape[0]} -> {sandbox_batch}")
print(f"loss kind: {sandbox_loss_kind}")
print(f"input grad norm: {x.grad.norm().item():.6f} -> {sandbox_x.grad.norm().item():.6f}")
print(f"saved grad ops: {len(trace.saved_grad_ops)} -> {len(sandbox_trace.saved_grad_ops)}")
print(f"grad fn calls: {len(trace.grad_fn_calls)} -> {len(sandbox_trace.grad_fn_calls)}")